# Jakarta Globe News Scraper

This notebook implements a web scraper for Jakarta Globe business news articles.

## Features
- Configurable date ranges (scrape articles from the last N days)
- Duplicate prevention (skips already-scraped articles)
- Optional full content extraction from article pages
- Saves results to JSON format

## How to Use
1. Run Cell 1 to install required packages
2. Run Cell 2 to import modules
3. Run Cell 3 to define the scraper class
4. Run Cell 4 to configure settings
5. Run Cell 5 to execute the scraper
6. Run Cell 6 (optional) to view results

In [ ]:
# Install required packages
!pip install requests beautifulsoup4 lxml -q
print("Packages installed successfully!")

---
## Cell 2: Imports
Import all necessary modules for the scraper.

In [ ]:
import argparse
import json
import logging
import os
import re
import sys
from datetime import datetime, timedelta
from typing import Optional
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

print("All modules imported successfully!")

---
## Cell 3: JakartaGlobeScraper Class
Complete scraper implementation with all methods.

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)


class JakartaGlobeScraper:
    """Scraper for Jakarta Globe news articles."""

    def __init__(self, days_back: int = 2, output_file: str = 'news_data.json',
                 base_url: str = 'https://jakartaglobe.id/search/business/1',
                 fetch_content: bool = False):
        """
        Initialize the scraper.
        
        Args:
            days_back: Number of days back to scrape
            output_file: Output JSON file path
            base_url: Base URL to start scraping from
            fetch_content: Whether to fetch full article content
        """
        self.days_back = days_back
        self.output_file = output_file
        self.base_url = base_url
        self.fetch_content = fetch_content
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })

        # Calculate cutoff date
        self.cutoff_date = datetime.now() - timedelta(days=days_back)
        self.cutoff_date = self.cutoff_date.replace(hour=0, minute=0, second=0, microsecond=0)

        # Load existing URLs to prevent duplicates
        self.existing_urls = self._load_existing_urls()

        logger.info(f"Initialized scraper with days_back={days_back}, cutoff_date={self.cutoff_date.date()}")
        logger.info(f"Loaded {len(self.existing_urls)} existing URLs")

    def _load_existing_urls(self) -> set:
        """Load existing URLs from output JSON files to prevent duplicates."""
        urls = set()

        if os.path.exists(self.output_file):
            try:
                with open(self.output_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    for item in data:
                        if 'url' in item:
                            urls.add(item['url'])
                logger.info(f"Loaded {len(urls)} URLs from {self.output_file}")
            except (json.JSONDecodeError, IOError) as e:
                logger.warning(f"Could not load {self.output_file}: {e}")

        return urls

    def _parse_relative_date(self, date_text: str) -> Optional[datetime]:
        """
        Parse relative date text like '2 hours ago', 'Yesterday', 'Feb 24, 2026'.
        
        Args:
            date_text: Raw date text from the webpage
            
        Returns:
            datetime object or None if parsing fails
        """
        date_text = date_text.strip().lower()
        now = datetime.now()

        # Handle "X minutes ago", "X hours ago", "X days ago"
        match = re.search(r'(\d+)\s+(minute|hour|day|week|month|year)s?\s+ago', date_text)
        if match:
            num = int(match.group(1))
            unit = match.group(2)
            if unit == 'minute':
                return now - timedelta(minutes=num)
            elif unit == 'hour':
                return now - timedelta(hours=num)
            elif unit == 'day':
                return now - timedelta(days=num)
            elif unit == 'week':
                return now - timedelta(weeks=num)
            elif unit == 'month':
                return now - timedelta(days=num * 30)
            elif unit == 'year':
                return now - timedelta(days=num * 365)

        # Handle "Yesterday"
        if 'yesterday' in date_text:
            return now - timedelta(days=1)

        # Handle "Today"
        if 'today' in date_text:
            return now

        # Handle formats like "Feb 24, 2026" or "February 24, 2026"
        try:
            # Try various date formats
            for fmt in ['%b %d, %Y', '%B %d, %Y', '%d %b %Y', '%d %B %Y']:
                try:
                    return datetime.strptime(date_text, fmt)
                except ValueError:
                    continue
        except Exception:
            pass

        return None

    def _get_page_url(self, page_num: int) -> str:
        """Get URL for a specific page number."""
        base = self.base_url.rsplit('/', 1)[0]
        return f"{base}/{page_num}"

    def _fetch_page(self, url: str) -> Optional[BeautifulSoup]:
        """Fetch and parse a page."""
        try:
            logger.info(f"Fetching: {url}")
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'lxml')
        except requests.RequestException as e:
            logger.error(f"Failed to fetch {url}: {e}")
            return None

    def _parse_article_listing(self, soup: BeautifulSoup) -> list:
        """
        Parse article listings from a page.
        
        Args:
            soup: BeautifulSoup object of the page
            
        Returns:
            List of article dictionaries
        """
        articles = []

        # Find article elements - Jakarta Globe uses div with class 'row mb-4 position-relative'
        article_elements = soup.find_all('div', class_='row mb-4 position-relative')

        for elem in article_elements:
            try:
                # Find title from h4 with class 'text-truncate-2-lines'
                title_elem = elem.find('h4', class_='text-truncate-2-lines')
                if not title_elem:
                    continue

                title = title_elem.get_text(strip=True)

                # Find URL from a with class 'stretched-link'
                link_elem = elem.find('a', class_='stretched-link')
                if not link_elem:
                    continue

                relative_url = link_elem.get('href', '')
                if not relative_url:
                    continue

                # Prepend base URL to relative URL
                url = urljoin('https://jakartaglobe.id', relative_url)

                # Skip if already exists
                if url in self.existing_urls:
                    logger.debug(f"Skipping duplicate: {url}")
                    continue

                # Extract date from relative time text (span with class 'text-muted small')
                date_elem = elem.find('span', class_='text-muted small')
                date_text = date_elem.get_text(strip=True) if date_elem else ''
                date = self._parse_relative_date(date_text)
                if not date:
                    continue

                # Extract category from span with class 'id-cat'
                category_elem = elem.find('span', class_='id-cat')
                category = category_elem.get_text(strip=True) if category_elem else 'Business'

                article = {
                    'title': title,
                    'url': url,
                    'date': date.strftime('%Y-%m-%d'),
                    'category': category,
                    'is_existing': False,
                    'scraped_at': datetime.now().isoformat()
                }

                articles.append(article)

            except Exception as e:
                logger.warning(f"Error parsing article element: {e}")
                continue

        return articles

    def _fetch_article_content(self, url: str) -> dict:
        """
        Fetch full content from an article page.
        
        Args:
            url: Article URL to fetch
            
        Returns:
            Dictionary with content, summary, published_time, categories, subcategory, tags
        """
        content_data = {
            'content': '',
            'summary': '',
            'published_time': '',
            'categories': [],
            'subcategory': '',
            'tags': []
        }

        try:
            soup = self._fetch_page(url)
            if not soup:
                return content_data

            # Extract main content - try multiple common selectors
            content_elem = (
                soup.find('div', class_=re.compile('article-body|entry-content|post-content|content-body', re.I)) or
                soup.find('article') or
                soup.find('div', class_=re.compile('detail|main-content', re.I))
            )

            if content_elem:
                # Get all paragraphs
                paragraphs = content_elem.find_all('p')
                content_text = ' '.join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
                content_data['content'] = content_text

                # First paragraph as summary
                if paragraphs:
                    content_data['summary'] = paragraphs[0].get_text(strip=True)

            # Extract published time - try multiple selectors
            time_elem = (
                soup.find('time') or
                soup.find('span', class_=re.compile('date|time|published', re.I)) or
                soup.find('div', class_=re.compile('date|time', re.I))
            )
            if time_elem:
                content_data['published_time'] = time_elem.get_text(strip=True)

            # Extract categories from breadcrumb or category links
            cat_elems = (
                soup.find_all('a', class_=re.compile('category|breadcrumb', re.I)) or
                soup.find_all('span', class_=re.compile('category', re.I))
            )
            content_data['categories'] = [c.get_text(strip=True) for c in cat_elems[:3] if c.get_text(strip=True)]

            # Extract tags
            tag_elems = soup.find_all('a', href=re.compile('/tag/'))
            content_data['tags'] = [t.get_text(strip=True) for t in tag_elems if t.get_text(strip=True)]

            logger.info(f"Fetched content for: {url[:60]}...")

        except Exception as e:
            logger.error(f"Error fetching content for {url}: {e}")

        return content_data

    def _save_articles(self, articles: list):
        """Append articles to JSON file."""
        # Load existing data
        existing_data = []

        if os.path.exists(self.output_file):
            try:
                with open(self.output_file, 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
            except (json.JSONDecodeError, IOError):
                pass

        # Append new data
        existing_data.extend(articles)

        # Save articles
        with open(self.output_file, 'w', encoding='utf-8') as f:
            json.dump(existing_data, f, ensure_ascii=False, indent=2)
        logger.info(f"Saved {len(articles)} articles to {self.output_file}")

    def scrape(self) -> int:
        """
        Main scraping method.
        
        Returns:
            Number of new articles found
        """
        all_new_articles = []
        max_pages = 100
        page = 1
        stop_pagination = False

        logger.info(f"Starting scrape from {self.base_url}")

        while page <= max_pages and not stop_pagination:
            page_url = self._get_page_url(page)
            soup = self._fetch_page(page_url)

            if not soup:
                logger.warning(f"Failed to fetch page {page}, stopping")
                break

            articles = self._parse_article_listing(soup)

            if not articles:
                logger.info(f"No articles found on page {page}, stopping")
                break

            # Check if all articles are older than cutoff
            all_old = True
            new_articles = []

            for article in articles:
                article_date = datetime.strptime(article['date'], '%Y-%m-%d')

                if article_date >= self.cutoff_date:
                    all_old = False
                    new_articles.append(article)

            if all_old:
                logger.info(f"All articles on page {page} are older than cutoff, stopping")
                break

            logger.info(f"Found {len(new_articles)} new articles on page {page}")

            # Fetch content if requested
            if self.fetch_content:
                for article in new_articles:
                    content_data = self._fetch_article_content(article['url'])
                    article.update(content_data)

            all_new_articles.extend(new_articles)

            # Update existing URLs
            for article in new_articles:
                self.existing_urls.add(article['url'])

            page += 1

        # Save results
        if all_new_articles:
            self._save_articles(all_new_articles)
            logger.info(f"Scraping complete. Total new articles: {len(all_new_articles)}")
        else:
            logger.info("No new articles found")

        return len(all_new_articles)

print("JakartaGlobeScraper class defined successfully!")

---
## Cell 4: Configuration
Set your scraping parameters here. Modify these values to customize the scraper behavior.

In [ ]:
# Configuration variables - modify these as needed

# Number of days back to scrape articles from
DAYS_BACK = 2

# Output JSON file path
OUTPUT_FILE = 'news_data.json'

# Base URL to scrape (Jakarta Globe business section)
BASE_URL = 'https://jakartaglobe.id/search/business/1'

# Whether to fetch full article content from each page
# Set to True to get full article text, summary, tags, etc.
# Note: This will take longer as it fetches each article page
FETCH_CONTENT = False

print("Configuration:")
print(f"  Days back: {DAYS_BACK}")
print(f"  Output file: {OUTPUT_FILE}")
print(f"  Base URL: {BASE_URL}")
print(f"  Fetch content: {FETCH_CONTENT}")

---
## Cell 5: Run Scraper
Create a scraper instance and run the scraping process.

In [ ]:
# Create scraper instance with the configuration
scraper = JakartaGlobeScraper(
    days_back=DAYS_BACK,
    output_file=OUTPUT_FILE,
    base_url=BASE_URL,
    fetch_content=FETCH_CONTENT
)

# Run the scraper
print("\n" + "="*50)
print("Starting Jakarta Globe Scraper...")
print("="*50 + "\n")

count = scraper.scrape()

# Print summary
print("\n" + "="*50)
print("Scraping complete!")
print(f"New articles found: {count}")
print(f"Output file: {OUTPUT_FILE}")
print("="*50)

---
## Cell 6: View Results (Optional)
Load and display the scraped data in a nice format.

In [ ]:
# Load and display scraped data
import pandas as pd

if os.path.exists(OUTPUT_FILE):
    # Load the JSON data
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        articles = json.load(f)
    
    print(f"Total articles in {OUTPUT_FILE}: {len(articles)}\n")
    
    if articles:
        # Display as DataFrame for nice formatting
        df = pd.DataFrame(articles)
        
        # Show basic info
        print("Articles by date:")
        print(df['date'].value_counts().sort_index())
        print("\n")
        
        print("Articles by category:")
        print(df['category'].value_counts())
        print("\n")
        
        # Show sample of articles
        print("Sample articles (most recent):")
        print("-"*80)
        
        # Display columns to show (exclude some verbose fields)
        display_cols = ['title', 'date', 'category', 'url']
        if 'summary' in df.columns:
            display_cols.append('summary')
        
        # Show last 5 articles
        for idx, article in enumerate(articles[-5:], 1):
            print(f"\n{idx}. {article.get('title', 'N/A')}")
            print(f"   Date: {article.get('date', 'N/A')}")
            print(f"   Category: {article.get('category', 'N/A')}")
            print(f"   URL: {article.get('url', 'N/A')}")
            if article.get('summary'):
                summary = article['summary'][:150] + "..." if len(article['summary']) > 150 else article['summary']
                print(f"   Summary: {summary}")
            print("-"*80)
    else:
        print("No articles found in the output file.")
else:
    print(f"Output file {OUTPUT_FILE} does not exist yet. Run the scraper first.")

---
## Additional: View Raw JSON
View the raw JSON output if needed.

In [ ]:
# Display raw JSON (first 2 articles)
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        articles = json.load(f)
    
    if articles:
        print("Raw JSON (first 2 articles):\n")
        print(json.dumps(articles[:2], indent=2, ensure_ascii=False))
    else:
        print("No articles to display.")
else:
    print(f"Output file {OUTPUT_FILE} does not exist yet.")